
## File 04 — Retry and Error Handling:


This notebook implements reusable error-handling mechanisms for your E-Commerce Lakehouse pipeline.

It covers:

- Exception handling.
- Automatic retry.
- Exponential backoff.
- Retryable and non-retryable errors.
- Task-level execution monitoring.
- Recovery logging.
- Bronze, Silver and Gold task examples.

In [0]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone
import time
import uuid
import logging
import random
import traceback


# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

LOG_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "pipeline_execution_logs"
)

ERROR_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "pipeline_error_handling_report"
)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{GOLD_SCHEMA}"
)


# ============================================================
# 3. PYTHON LOGGER CONFIGURATION
# ============================================================

logger = logging.getLogger(
    "ecommerce_retry_handler"
)

logger.setLevel(logging.INFO)

if not logger.handlers:

    handler = logging.StreamHandler()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    handler.setFormatter(formatter)

    logger.addHandler(handler)


# ============================================================
# 4. ERROR REPORT TABLE SCHEMA
# ============================================================

error_schema = StructType([

    StructField("error_id", StringType(), False),

    StructField("run_id", StringType(), True),

    StructField("pipeline_name", StringType(), True),

    StructField("task_name", StringType(), True),

    StructField("dataset_name", StringType(), True),

    StructField("attempt_number", IntegerType(), True),

    StructField("max_attempts", IntegerType(), True),

    StructField("error_type", StringType(), True),

    StructField("error_message", StringType(), True),

    StructField("retryable", BooleanType(), True),

    StructField("execution_status", StringType(), True),

    StructField("retry_delay_seconds", DoubleType(), True),

    StructField("logged_at", TimestampType(), True)

])


# ============================================================
# 5. CREATE ERROR REPORT TABLE
# ============================================================

if not spark.catalog.tableExists(ERROR_TABLE):

    (

        spark.createDataFrame(
            [],
            error_schema
        )

        .write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(ERROR_TABLE)

    )


# ============================================================
# 6. GENERATE RUN ID
# ============================================================

RUN_ID = str(uuid.uuid4())

PIPELINE_NAME = "ecommerce_lakehouse_pipeline"


# ============================================================
# 7. DEFINE RETRYABLE EXCEPTIONS
# ============================================================

RETRYABLE_EXCEPTION_NAMES = {

    "TimeoutError",

    "ConnectionError",

    "ConnectionResetError",

    "ConnectionAbortedError",

    "BrokenPipeError"

}


# ============================================================
# 8. CLASSIFY ERROR
# ============================================================

def is_retryable_error(error):

    error_name = type(error).__name__

    error_message = str(error).lower()

    if error_name in RETRYABLE_EXCEPTION_NAMES:

        return True

    retryable_keywords = [

        "timeout",

        "temporarily unavailable",

        "connection reset",

        "connection refused",

        "service unavailable",

        "rate limit",

        "too many requests",

        "concurrent modification",

        "temporarily overloaded"

    ]

    return any(

        keyword in error_message

        for keyword in retryable_keywords

    )


# ============================================================
# 9. SAVE ERROR DETAILS TO DELTA
# ============================================================

def save_error_record(

    task_name,

    dataset_name,

    attempt_number,

    max_attempts,

    error,

    retryable,

    execution_status,

    retry_delay_seconds=0

):

    record = {

        "error_id": str(uuid.uuid4()),

        "run_id": RUN_ID,

        "pipeline_name": PIPELINE_NAME,

        "task_name": task_name,

        "dataset_name": dataset_name,

        "attempt_number": int(attempt_number),

        "max_attempts": int(max_attempts),

        "error_type": type(error).__name__,

        "error_message": str(error)[:4000],

        "retryable": bool(retryable),

        "execution_status": execution_status,

        "retry_delay_seconds": float(
            retry_delay_seconds
        ),

        "logged_at": (
            datetime.now(timezone.utc).replace(
                tzinfo=None
            )
        )

    }

    error_df = spark.createDataFrame(

        [record],

        schema=error_schema

    )

    (

        error_df

        .write

        .format("delta")

        .mode("append")

        .saveAsTable(ERROR_TABLE)

    )


# ============================================================
# 10. EXPONENTIAL BACKOFF FUNCTION
# ============================================================

def calculate_retry_delay(

    attempt_number,

    base_delay_seconds=5,

    max_delay_seconds=60

):

    delay = (

        base_delay_seconds

        * (2 ** (attempt_number - 1))

    )

    # Add jitter to reduce simultaneous retry attempts.

    jitter = random.uniform(0, 1)

    return min(

        delay + jitter,

        max_delay_seconds

    )


# ============================================================
# 11. REUSABLE RETRY EXECUTOR
# ============================================================

def execute_with_retry(

    task_function,

    task_name,

    dataset_name=None,

    max_attempts=3,

    base_delay_seconds=5,

    max_delay_seconds=60

):

    if max_attempts < 1:

        raise ValueError(
            "max_attempts must be at least 1."
        )

    logger.info(
        f"Starting task: {task_name}"
    )

    for attempt in range(

        1,

        max_attempts + 1

    ):

        try:

            result = task_function()

            logger.info(

                f"Task succeeded: {task_name}, "

                f"attempt: {attempt}"

            )

            return {

                "task_name": task_name,

                "dataset_name": dataset_name,

                "status": "SUCCESS",

                "attempt_number": attempt,

                "result": result

            }

        except Exception as error:

            retryable = is_retryable_error(
                error
            )

            if not retryable:

                execution_status = "FAILED_PERMANENT"

                save_error_record(

                    task_name=task_name,

                    dataset_name=dataset_name,

                    attempt_number=attempt,

                    max_attempts=max_attempts,

                    error=error,

                    retryable=False,

                    execution_status=execution_status

                )

                logger.exception(
                    f"Permanent failure: {task_name}"
                )

                raise

            if attempt == max_attempts:

                execution_status = "FAILED_MAX_RETRIES"

                save_error_record(

                    task_name=task_name,

                    dataset_name=dataset_name,

                    attempt_number=attempt,

                    max_attempts=max_attempts,

                    error=error,

                    retryable=True,

                    execution_status=execution_status

                )

                logger.exception(

                    f"Maximum retries reached: {task_name}"

                )

                raise

            retry_delay = calculate_retry_delay(

                attempt_number=attempt,

                base_delay_seconds=base_delay_seconds,

                max_delay_seconds=max_delay_seconds

            )

            save_error_record(

                task_name=task_name,

                dataset_name=dataset_name,

                attempt_number=attempt,

                max_attempts=max_attempts,

                error=error,

                retryable=True,

                execution_status="RETRY_SCHEDULED",

                retry_delay_seconds=retry_delay

            )

            logger.warning(

                f"Task failed: {task_name}. "

                f"Retrying in {retry_delay:.2f} seconds."

            )

            time.sleep(retry_delay)

    raise RuntimeError(
        f"Unexpected retry execution state: {task_name}"
    )


# ============================================================
# 12. BRONZE TASK EXAMPLE
# ============================================================

def bronze_customer_task():

    table_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}.customers"
    )

    if not spark.catalog.tableExists(table_name):

        raise ValueError(
            f"Bronze table not found: {table_name}"
        )

    record_count = spark.table(
        table_name
    ).count()

    return {

        "layer": "BRONZE",

        "records_read": record_count,

        "records_processed": record_count

    }


# ============================================================
# 13. SILVER TASK EXAMPLE
# ============================================================

def silver_customer_task():

    table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.silver_customers"
    )

    if not spark.catalog.tableExists(table_name):

        raise ValueError(
            f"Silver table not found: {table_name}"
        )

    record_count = spark.table(
        table_name
    ).count()

    return {

        "layer": "SILVER",

        "records_read": record_count,

        "records_processed": record_count

    }


# ============================================================
# 14. GOLD TASK EXAMPLE
# ============================================================

def gold_sales_task():

    table_name = (
        f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_sales"
    )

    if not spark.catalog.tableExists(table_name):

        raise ValueError(
            f"Gold table not found: {table_name}"
        )

    record_count = spark.table(
        table_name
    ).count()

    return {

        "layer": "GOLD",

        "records_read": record_count,

        "records_processed": record_count

    }


# ============================================================
# 15. EXECUTE PIPELINE TASKS
# ============================================================

pipeline_tasks = [

    {

        "task_name": "bronze_customer_check",

        "dataset_name": "customers",

        "function": bronze_customer_task

    },

    {

        "task_name": "silver_customer_check",

        "dataset_name": "customers",

        "function": silver_customer_task

    },

    {

        "task_name": "gold_daily_sales_check",

        "dataset_name": None,

        "function": gold_sales_task

    }

]


task_results = []

for task in pipeline_tasks:

    try:

        result = execute_with_retry(

            task_function=task["function"],

            task_name=task["task_name"],

            dataset_name=task["dataset_name"],

            max_attempts=3,

            base_delay_seconds=5,

            max_delay_seconds=30

        )

        task_results.append(result)

    except Exception as error:

        task_results.append({

            "task_name": task["task_name"],

            "dataset_name": task["dataset_name"],

            "status": "FAILED",

            "error_message": str(error)

        })

        logger.error(

            f"Pipeline task stopped: {task['task_name']}"

        )

        # Stop dependent tasks after a failure.

        break


# ============================================================
# 16. DISPLAY EXECUTION RESULTS
# ============================================================

for result in task_results:

    print(result)


# ============================================================
# 17. DISPLAY ERROR HANDLING REPORT
# ============================================================

error_report_df = (

    spark.table(ERROR_TABLE)

    .filter(
        F.col("run_id") == RUN_ID
    )

    .orderBy(
        F.col("logged_at").desc()
    )

)

display(error_report_df)


# ============================================================
# 18. EXECUTION SUMMARY
# ============================================================

successful_tasks = sum(

    1 for result in task_results

    if result["status"] == "SUCCESS"

)

failed_tasks = sum(

    1 for result in task_results

    if result["status"] == "FAILED"

)

print("\n" + "=" * 65)

print("RETRY AND ERROR HANDLING SUMMARY")

print("=" * 65)

print(f"Run ID: {RUN_ID}")

print(f"Successful tasks: {successful_tasks}")

print(f"Failed tasks: {failed_tasks}")

print(f"Total executed tasks: {len(task_results)}")

print(f"Error report table: {ERROR_TABLE}")

print("=" * 65)


# ============================================================
# 19. FINAL PIPELINE STATUS
# ============================================================

if failed_tasks > 0:

    print("PIPELINE STATUS: FAILED")

    raise RuntimeError(
        "Pipeline execution failed. "
        "Review the error handling report."
    )

else:

    print("PIPELINE STATUS: SUCCESS")

print(
    "RETRY AND ERROR HANDLING COMPLETED."
)



## View retry attempts

In [0]:
%sql

SELECT
    run_id,
    task_name,
    attempt_number,
    max_attempts,
    error_type,
    retryable,
    execution_status,
    retry_delay_seconds,
    logged_at
FROM workspace.ecommerce_gold.pipeline_error_handling_report
ORDER BY logged_at DESC;


## Count retry attempts by task

In [0]:
%sql

SELECT
    task_name,
    COUNT(*) AS total_error_events,
    SUM(
        CASE
            WHEN retryable = true THEN 1
            ELSE 0
        END
    ) AS retryable_errors,
    SUM(
        CASE
            WHEN execution_status = 'FAILED_MAX_RETRIES'
            THEN 1
            ELSE 0
        END
    ) AS exhausted_retries
FROM workspace.ecommerce_gold.pipeline_error_handling_report
GROUP BY task_name;